In [1]:
import duckdb
import os

print("DuckDB is ready!")

con = duckdb.connect()

print("Connection established!")

DuckDB is ready!
Connection established!


In [7]:
import duckdb
import os

# Location of the original dataset
csv_path = r"../data/raw/transactions_train.csv"

# Check whether the file exists
if os.path.exists(csv_path):
    print("Dataset found!")
    print(f"File size: {os.path.getsize(csv_path) / (1024**3):.2f} GB")
else:
    print("Dataset not found. Please check the file path.")

# Establish DuckDB connection
con = duckdb.connect()

print("DuckDB connection established!")

Dataset found!
File size: 3.25 GB
DuckDB connection established!


In [8]:
# Read the CSV using DuckDB
# This does NOT load the entire 3.2 GB file into Pandas.

transactions = con.read_csv(csv_path)

# Display column names
print("DATASET COLUMNS:")
print(transactions.columns)

# Display column data types
print("\nDATA TYPES:")
print(transactions.types)

# Display the first 5 records
print("\nFIRST 5 RECORDS:")
display(transactions.limit(5).df())

DATASET COLUMNS:
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']

DATA TYPES:
[DATE, VARCHAR, VARCHAR, DOUBLE, BIGINT]

FIRST 5 RECORDS:


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2


In [9]:
record_count = con.execute("""
    SELECT COUNT(*) 
    FROM read_csv_auto(?)
""", [csv_path]).fetchone()[0]

print(f"Total records: {record_count:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total records: 31,788,324


In [10]:
quality_summary = con.execute("""
    SELECT
        COUNT(*) AS total_records,
        MIN(t_dat) AS first_transaction_date,
        MAX(t_dat) AS last_transaction_date,

        COUNT(*) FILTER (WHERE t_dat IS NULL) AS missing_dates,
        COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customers,
        COUNT(*) FILTER (WHERE article_id IS NULL) AS missing_articles,
        COUNT(*) FILTER (WHERE price IS NULL) AS missing_prices,
        COUNT(*) FILTER (WHERE sales_channel_id IS NULL) AS missing_channels,

        COUNT(*) FILTER (WHERE price <= 0) AS non_positive_prices

    FROM read_csv_auto(?)
""", [csv_path]).df()

display(quality_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_records,first_transaction_date,last_transaction_date,missing_dates,missing_customers,missing_articles,missing_prices,missing_channels,non_positive_prices
0,31788324,2018-09-20,2020-09-22,0,0,0,0,0,0


In [11]:
duplicate_summary = con.execute("""
    SELECT
        SUM(record_count - 1) AS duplicate_records
    FROM (
        SELECT COUNT(*) AS record_count
        FROM read_csv_auto(?)
        GROUP BY
            t_dat,
            customer_id,
            article_id,
            price,
            sales_channel_id
        HAVING COUNT(*) > 1
    )
""", [csv_path]).df()

display(duplicate_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_records
0,2974905.0


In [12]:
duplicate_check = con.execute("""
    WITH grouped_transactions AS (
        SELECT
            t_dat,
            customer_id,
            article_id,
            price,
            sales_channel_id,
            COUNT(*) AS occurrences
        FROM read_csv_auto(?)
        GROUP BY
            t_dat,
            customer_id,
            article_id,
            price,
            sales_channel_id
    )

    SELECT
        COUNT(*) AS duplicate_groups,
        SUM(occurrences) AS records_in_duplicate_groups,
        SUM(occurrences - 1) AS extra_duplicate_records,
        MAX(occurrences) AS maximum_repetitions

    FROM grouped_transactions
    WHERE occurrences > 1
""", [csv_path]).df()

display(duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_groups,records_in_duplicate_groups,extra_duplicate_records,maximum_repetitions
0,2543908,5518813.0,2974905.0,569


In [15]:
articles_path = r"../data/raw/articles.csv"

# Check whether the file exists
import os

if os.path.exists(articles_path):
    print("Articles dataset found!")
    print(
        f"File size: "
        f"{os.path.getsize(articles_path) / (1024**2):.2f} MB"
    )
else:
    print("Articles dataset not found. Check the file path.")

# Inspect the dataset using DuckDB
articles = con.read_csv(articles_path)

print("\nCOLUMNS:")
print(articles.columns)

print("\nDATA TYPES:")
print(articles.types)

print("\nFIRST 5 RECORDS:")
display(articles.limit(5).df())

articles_count = con.execute("""
    SELECT COUNT(*)
    FROM read_csv_auto(?)
""", [articles_path]).fetchone()[0]

print(f"Total article records: {articles_count:,}")

Articles dataset found!
File size: 34.45 MB

COLUMNS:
['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']

DATA TYPES:
[VARCHAR, VARCHAR, VARCHAR, BIGINT, VARCHAR, VARCHAR, BIGINT, VARCHAR, VARCHAR, VARCHAR, BIGINT, VARCHAR, BIGINT, VARCHAR, BIGINT, VARCHAR, VARCHAR, VARCHAR, BIGINT, VARCHAR, BIGINT, VARCHAR, BIGINT, VARCHAR, VARCHAR]

FIRST 5 RECORDS:


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,09,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,0108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,09,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


Total article records: 105,542


In [19]:
import duckdb

# Create a separate connection for customer inspection
customer_con = duckdb.connect()

print("Customer inspection connection established!")

Customer inspection connection established!


In [20]:
# Display column names and data types
customer_data = customer_con.execute("""
    SELECT *
    FROM read_csv_auto(?)
    LIMIT 5
""", [customers_path]).df()

print("CUSTOMER COLUMNS:")
print(customer_data.columns.tolist())

print("\nDATA TYPES:")
print(customer_data.dtypes)

print("\nFIRST 5 RECORDS:")
display(customer_data)

CUSTOMER COLUMNS:
['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']

DATA TYPES:
customer_id                   str
FN                        float64
Active                    float64
club_member_status            str
fashion_news_frequency        str
age                         int64
postal_code                   str
dtype: object

FIRST 5 RECORDS:


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [22]:
article_quality = con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT article_id) AS unique_article_ids,

        COUNT(*) FILTER (
            WHERE article_id IS NULL
        ) AS missing_article_ids,

        COUNT(*) FILTER (
            WHERE prod_name IS NULL
        ) AS missing_product_names,

        COUNT(*) FILTER (
            WHERE product_type_name IS NULL
        ) AS missing_product_types,

        COUNT(*) FILTER (
            WHERE garment_group_name IS NULL
        ) AS missing_garment_groups

    FROM read_csv_auto(?)
""", [articles_path]).df()

display(article_quality)

,total_records,unique_article_ids,missing_article_ids,missing_product_names,missing_product_types,missing_garment_groups
0,105542,105542,0,0,0,0


In [21]:
customer_quality = customer_con.execute("""
    SELECT
        COUNT(*) AS total_records,

        COUNT(DISTINCT customer_id) AS unique_customer_ids,

        COUNT(*) FILTER (
            WHERE customer_id IS NULL
        ) AS missing_customer_ids,

        COUNT(*) FILTER (
            WHERE age IS NULL
        ) AS missing_ages

    FROM read_csv_auto(?)
""", [customers_path]).df()

display(customer_quality)

,total_records,unique_customer_ids,missing_customer_ids,missing_ages
0,1371980,1371980,0,15861


In [23]:
import duckdb

# Create a fresh connection for preprocessing preparation
prep_con = duckdb.connect()

print("Preprocessing connection established!")

Preprocessing connection established!


In [24]:
transaction_dates = prep_con.execute("""
    SELECT
        MIN(CAST(t_dat AS DATE)) AS first_transaction_date,
        MAX(CAST(t_dat AS DATE)) AS last_transaction_date,
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT CAST(t_dat AS DATE)) AS unique_transaction_days,
        COUNT(DISTINCT article_id) AS unique_articles,
        COUNT(DISTINCT customer_id) AS unique_customers

    FROM read_csv_auto(?)
""", [csv_path]).df()

display(transaction_dates)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_transaction_date,last_transaction_date,total_transactions,unique_transaction_days,unique_articles,unique_customers
0,2018-09-20,2020-09-22,31788324,734,104547,1362281


In [25]:
article_matching = prep_con.execute("""
    SELECT
        COUNT(DISTINCT t.article_id) AS transaction_article_ids,

        COUNT(DISTINCT CASE
            WHEN a.article_id IS NOT NULL
            THEN t.article_id
        END) AS matched_article_ids,

        COUNT(DISTINCT CASE
            WHEN a.article_id IS NULL
            THEN t.article_id
        END) AS unmatched_article_ids

    FROM read_csv_auto(?) t

    LEFT JOIN read_csv_auto(?) a
        ON t.article_id = a.article_id
""", [csv_path, articles_path]).df()

display(article_matching)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transaction_article_ids,matched_article_ids,unmatched_article_ids
0,104547,104547,0
